# Phase 5: KPI Calculation and Final Tableau Load Preparation

This notebook prepares the final datasets for Tableau dashboarding.
To avoid mixed data grains and ensure accurate aggregations, we split the output into two datasets:

### 1. Customer-Level Dataset (Primary)
**Purpose:** Churn analysis, customer segmentation, and lifetime value.
**Grain:** One row per `customer_unique_id`.
**Key Metrics:** Total orders, total revenue, AOV, average delivery time, recency, churn flag, etc.

### 2. Order-Level Dataset (Optional/Secondary)
**Purpose:** Time-series trends, category analysis, and operational performance.
**Grain:** One row per `order_id`.

## Dashboard Purpose Alignment
This dataset is designed to explicitly support:
- **Customer segmentation**
- **Churn analysis** (via `churn_flag`)
- **Revenue distribution**
- **Category performance**
- **Operational insights** (delivery times)

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load the cleaned dataset
df = pd.read_csv('../data/processed/final_dataset.csv')

## 1. Order-Level Dataset
Keep order-level data separate for trends and category analysis.

In [ ]:
df['total_order_value'] = df['price'] + df['freight_value']
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])
df['is_late_delivery'] = np.where(df['order_delivered_customer_date'] > df['order_estimated_delivery_date'], 1, 0)

# Select useful order-level columns
order_cols = [
    'order_id', 'customer_unique_id', 'order_purchase_timestamp', 
    'total_order_value', 'is_late_delivery', 'product_category_name_english', 
    'review_score', 'order_status'
]
order_df = df[[col for col in order_cols if col in df.columns]].drop_duplicates()
order_df.to_csv('../data/processed/tableau_order_level.csv', index=False)
print("Order-level dataset exported.")

## 2. Customer-Level Dataset
Calculate KPIs explicitly per customer.

In [ ]:
# Convert purchase timestamp to datetime to calculate recency
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
max_date = df['order_purchase_timestamp'].max()

# Calculate Recency per customer
recency_df = df.groupby('customer_unique_id')['order_purchase_timestamp'].max().reset_index()
recency_df['recency'] = (max_date - recency_df['order_purchase_timestamp']).dt.days

# Calculate Customer KPIs
customer_kpis = df.groupby('customer_unique_id').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('total_payment_value', 'sum'),
    avg_order_value=('total_payment_value', 'mean'),
    avg_delivery_time=('delivery_time', 'mean'),
    avg_review_score=('review_score', 'mean')
).reset_index()

# Merge recency
customer_kpis = customer_kpis.merge(recency_df[['customer_unique_id', 'recency']], on='customer_unique_id')

# Define Churn Flag: 1 if recency > 90 days, else 0
customer_kpis['churn_flag'] = np.where(customer_kpis['recency'] > 90, 1, 0)


In [ ]:
# Calculate Top Category efficiently
top_cat = df.groupby(['customer_unique_id', 'product_category_name_english']).size().reset_index(name='count')
top_cat = top_cat.sort_values('count', ascending=False).drop_duplicates('customer_unique_id')
top_cat = top_cat.rename(columns={'product_category_name_english': 'top_category'})[['customer_unique_id', 'top_category']]

# Calculate Preferred Payment efficiently
pref_pay = df.groupby(['customer_unique_id', 'dominant_payment_type']).size().reset_index(name='count')
pref_pay = pref_pay.sort_values('count', ascending=False).drop_duplicates('customer_unique_id')
pref_pay = pref_pay.rename(columns={'dominant_payment_type': 'preferred_payment_type'})[['customer_unique_id', 'preferred_payment_type']]

customer_kpis = customer_kpis.merge(top_cat, on='customer_unique_id', how='left')
customer_kpis = customer_kpis.merge(pref_pay, on='customer_unique_id', how='left')

# Add location
location_df = df.groupby('customer_unique_id').agg(
    location_city=('customer_city', 'first'),
    location_state=('customer_state', 'first')
).reset_index()

customer_kpis = customer_kpis.merge(location_df, on='customer_unique_id', how='left')


In [ ]:
# Keep ONLY what is useful
final_cols = [
    'customer_unique_id', 'total_orders', 'total_revenue', 'avg_order_value', 
    'avg_delivery_time', 'avg_review_score', 'recency', 'churn_flag', 
    'preferred_payment_type', 'top_category', 'location_city', 'location_state'
]
customer_kpis = customer_kpis[[col for col in final_cols if col in customer_kpis.columns]]

# Export final customer-level dataset
customer_kpis.to_csv('../data/processed/tableau_customer_level.csv', index=False)
print("Customer-level dataset exported. Ready for Tableau Dashboarding.")
